## Medicine Agentic AI

In [ ]:
!pip -q install -U "requests==2.32.4" \
  "langchain>=0.2,<0.3" "langchain-core>=0.2,<0.3" "langchain-community>=0.2,<0.3" \
  langchain-groq serpapi python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.26.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
opencv-pyth

In [ ]:
import os, getpass
from secret_key import  serpapi_key, groq_api_key

# Groq API key (for llama-3.3-70b-versatile)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass(groq_api_key)

# SerpAPI key (web search via Google)
if "SERPAPI_API_KEY" not in os.environ:
    os.environ["SERPAPI_API_KEY"] = getpass.getpass(serpapi_key)

In [ ]:
!pip install google-search-results

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=7c7b9b5b5c5bf65ce832195589b5a0bca9856d00264d6902ce6812c3a019e6f5
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


In [ ]:
from serpapi import GoogleSearch
print("GoogleSearch OK:", GoogleSearch is not None)

GoogleSearch OK: True


In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate


# 1) LLM: Groq (llama-3.3-70b-versatile)
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.2,      # keep answers factual & consistent
    max_retries=2,
)

# 2) Tool: SerpAPI web search
search = SerpAPIWrapper(
    serpapi_api_key=os.environ["SERPAPI_API_KEY"],
    params={
        "engine": "google",
        "hl": "en",
        "gl": "us",
        "num": 6,  # top results; adjust as needed
    },
)

web_search_tool = Tool(
    name="web_search",
    func=search.run,
    description=(
        "Search the web for authoritative medical information. "
        "Use this for: indications, benefits, risks/side effects, contraindications, precautions. "
        "Prefer reputable sources like FDA, NHS, MedlinePlus, WHO, NICE, Drugs.com, RxList, Mayo Clinic."
    ),
)

tools = [web_search_tool]

# 3) ReAct-style prompt (self-contained — no hub dependency)
react_prompt = PromptTemplate(
    input_variables=["input", "agent_scratchpad", "tools", "tool_names"],
    template="""
You are a cautious medical information assistant. The user will provide a medicine name.
Your task:
1) Explain what the medicine is used for (indications / therapeutic use).
2) List advantages (benefits, when appropriate).
3) List disadvantages (risks, side effects, important cautions, black-box warnings if any).

**Rules**
- ALWAYS consult the web_search tool to verify details for each medicine.
- Prefer authoritative sources (FDA, NHS, MedlinePlus, WHO, NICE, Drugs.com, RxList, Mayo Clinic).
- Keep the tone clear and concise. Use bullet points.
- Provide inline citations like [Source: Site/Title] and end with a short Sources section listing URLs.
- If multiple formulations/indications exist, note the most common and mention variability.
- If evidence conflicts, say so.
- Include a brief safety disclaimer at the end: "This is educational information, not medical advice."

You can use these tools:
{tools}

Use this reasoning format:

Question: {input}
Thought: think about what to do next
Action: one of [{tool_names}]
Action Input: the input for the action
Observation: result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I have enough information
Final Answer:
- **What it’s used for**:
  - ...
- **Advantages**:
  - ...
- **Disadvantages / Risks**:
  - ...
- **Sources**:
  - URL 1
  - URL 2

Question: {input}
{agent_scratchpad}
""".strip(),
)

# 4) Create the agent and executor
agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,               # set to False to suppress ReAct trace
    handle_parsing_errors=True,
    max_iterations=6,           # allow several tool calls if needed
    early_stopping_method="force",
    #early_stopping_method="generate",
)


In [ ]:
def explain_medicine(medicine_name: str) -> str:
    """Query the agent and return a formatted explanation."""
    resp = agent_executor.invoke({"input": medicine_name})
    return resp["output"]

print(explain_medicine("atorvastatin"))


Thought: To provide accurate information about atorvastatin, I need to search for its indications, benefits, and risks.

Action: web_search
Action Input: atorvastatin indications benefits risks['While statins are effective and safe for most people, they have been linked to muscle pain, digestive problems, and mental fuzziness in some people.', 'Atorvastatin is a medication primarily used to manage and treat dyslipidemias and prevent cardiovascular disease. This activity focuses on the ...', "Atorvastatin helps lower LDL cholesterol ('bad cholesterol') and triglyceride levels, and increases HDL cholesterol ('good cholesterol') levels, ...", 'Atorvastatin is used to. reduce the risk of heart attack and stroke; decrease the amount of cholesterol (a fat-like substance that can build up and clog ...', "Atorvastatin (Lipitor, Atorvaliq) is a statin medication used to treat high cholesterol. It's also used to lower the risk of heart attack or stroke in certain ...", "It is used to lower chole

In [ ]:
# Core imports
import os, json
from langchain_groq import ChatGroq
from langchain_core.tools import Tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

# --- Ensure the correct SerpAPI client is present and NOT shadowed ---
# The ONLY correct client is 'google-search-results', which provides `from serpapi import GoogleSearch`.
# If a different 'serpapi' package or a local 'serpapi.py' exists, imports will fail.

# Quick sanity check (no error means OK)
from serpapi import GoogleSearch  # provided by google-search-results

# LangChain SerpAPI wrapper (uses the above client internally)
from langchain_community.utilities import SerpAPIWrapper

# --- LLM: Groq llama-3.3-70b-versatile ---
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.2,   # lower temperature for consistent structured answers
    max_retries=2,
)

# --- SerpAPI tool returning JSON (titles + URLs) for cleaner citations ---
search = SerpAPIWrapper(
    serpapi_api_key=os.environ["SERPAPI_API_KEY"],
    params={"engine": "google", "hl": "en", "gl": "us", "num": 6},
)

def web_search_json(query: str):
    """
    Use SerpAPI to return JSON results with titles and links.
    This is easier for the model to turn into clean, citable bullets.
    """
    try:
        results = search.results(query, num=6)
        # Keep only useful fields
        payload = []
        for item in results.get("organic_results", []):
            payload.append({
                "title": item.get("title"),
                "link": item.get("link"),
                "snippet": item.get("snippet")
            })
        return json.dumps(payload, ensure_ascii=False, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})

web_search_tool = Tool(
    name="web_search",
    func=web_search_json,
    description=(
        "Search the web for authoritative medical information. Returns JSON with title, link, snippet. "
        "Use for indications, benefits, risks/side effects, contraindications, precautions. "
        "Prefer FDA, NHS, MedlinePlus, WHO, NICE, Drugs.com, RxList, Mayo Clinic."
    ),
)

tools = [web_search_tool]

# --- ReAct prompt (tightened to reduce parser errors) ---
react_prompt = PromptTemplate(
    input_variables=["input", "agent_scratchpad", "tools", "tool_names"],
    template="""
You are a cautious medical information assistant. The user provides a medicine name.

Your tasks:
1) Explain what the medicine is used for (indications / therapeutic use).
2) List advantages (benefits).
3) List disadvantages (risks, side effects, cautions; include black-box warnings if any).

Rules:
- ALWAYS call the web_search tool at least once to verify facts for each medicine.
- Prefer authoritative sources (FDA, NHS, MedlinePlus, WHO, NICE, Drugs.com, RxList, Mayo Clinic).
- Be concise. Use bullet points.
- Provide inline citations like [Source: Site/Title] within bullets and list URLs in a final Sources section.
- If multiple formulations/indications exist, note variability.
- If evidence conflicts, say so.
- End with the sentence: "This is educational information, not medical advice."

STRICT TOOL USE FORMAT:
Thought: describe what to do next
Action: one of [{tool_names}]
Action Input: the exact input for the action
Observation: result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I have enough information
Final Answer:
- **What it’s used for**:
  - ...
- **Advantages**:
  - ...
- **Disadvantages / Risks**:
  - ...
- **Sources**:
  - URL 1
  - URL 2

Never include anything after 'Final Answer:' other than the final content above.
If you encounter any formatting/parsing issues, STOP using tools and immediately produce 'Final Answer:' in the required structure.

You can use these tools:
{tools}

Question: {input}
{agent_scratchpad}
""".strip(),
)

# --- Construct the agent + executor ---
agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,  # turn off step-by-step trace; set True to debug agent Thoughts/Actions
    handle_parsing_errors=(
        "PARSING_ERROR: Stop tool use and produce 'Final Answer:' with the required sections."
    ),
    max_iterations=6,
    early_stopping_method="force",  # IMPORTANT: some 0.2.x builds don't support 'generate'
)

def explain_medicine(medicine_name: str) -> str:
    """
    Run the agent and return the formatted explanation string:
    - What it’s used for
    - Advantages
    - Disadvantages / Risks
    - Sources
    """
    try:
        resp = agent_executor.invoke({"input": medicine_name})
        return resp.get("output", str(resp))
    except Exception as e:
        # Soft fallback to a single LLM call if the agent loop fails
        rescue_prompt = f"""
You attempted to analyze the medicine '{medicine_name}' but the agent failed due to a parsing/format issue.
Without using any tools now, produce the 'Final Answer' sections exactly:

- **What it’s used for**:
  - ...
- **Advantages**:
  - ...
- **Disadvantages / Risks**:
  - ...
- **Sources**:
  - ...

Add the sentence: "This is educational information, not medical advice."
"""
        return llm.invoke(rescue_prompt).content

# --- Quick smoke test ---
#print(explain_medicine("atorvastatin"))
print(explain_medicine("dolo"))

- **What it’s used for**:
  - Dolo, commonly known as Paracetamol or Acetaminophen, is used to relieve pain and reduce fever [Source: MedlinePlus/Acetaminophen].
  - It is used to treat headaches, other minor aches and pains, and is a major ingredient in numerous cold and flu remedies [Source: NHS/Paracetamol].
- **Advantages**:
  - Effective in relieving mild to moderate pain [Source: Mayo Clinic/Acetaminophen].
  - Helps reduce fever [Source: Drugs.com/Acetaminophen].
  - Generally considered safe when used as directed [Source: FDA/Acetaminophen].
- **Disadvantages / Risks**:
  - Can cause liver damage if taken in excess [Source: MedlinePlus/Acetaminophen].
  - May interact with other medications [Source: RxList/Acetaminophen].
  - Black-box warning: risk of severe liver injury, especially with overdose [Source: FDA/Acetaminophen].
- **Sources**:
  - https://medlineplus.gov/druginfo/meds/a681004.html
  - https://www.nhs.uk/medicines/paracetamol/
  - https://www.mayoclinic.org/drugs-s